## FISPACT II legacy output

F4Enix wraps [pypact](https://github.com/fispact/pypact) to provide some higher level routines and to complement some of the missing feature.

## Parsing of pathways

```{eval-rst}
The complete API can be found at :py:class:`f4enix.output.fispact_legacy_out.PathwayCollection`
```

Unfortunately, reaction pathways are not supported yet in pypact, so a parser has been built directly in F4Enix

In [1]:
from f4enix.output.fispact_legacy_out import PathwayCollection
from pprint import pprint # just to print in a nicer way

pathways_collection = PathwayCollection.from_file('testSS.out')
pprint(pathways_collection.pathways)  # print the pathways in the collection

[Pathway(parent=FispactZaid(element='Mn', isotope=55, metastable=False),
         daughter=FispactZaid(element='Mn', isotope=56, metastable=False),
         perc=43.807,
         reactions=['(n,g)'],
         intermediates=None),
 Pathway(parent=FispactZaid(element='Fe', isotope=56, metastable=False),
         daughter=FispactZaid(element='Mn', isotope=56, metastable=False),
         perc=56.135,
         reactions=['(n,p)'],
         intermediates=None),
 Pathway(parent=FispactZaid(element='Cr', isotope=52, metastable=False),
         daughter=FispactZaid(element='V', isotope=52, metastable=False),
         perc=96.532,
         reactions=['(n,p)'],
         intermediates=None),
 Pathway(parent=FispactZaid(element='Mn', isotope=55, metastable=False),
         daughter=FispactZaid(element='V', isotope=52, metastable=False),
         perc=2.86,
         reactions=['(n,a)'],
         intermediates=None),
 Pathway(parent=FispactZaid(element='Si', isotope=28, metastable=False),
         da

In [2]:
# as it can see above, each pathway is represented as a dataclass that have
# different attributes that can be accessed directly
pathway = pathways_collection.pathways[13]
# a handy str method is provided to get a compact description of the pathway
print(pathway)
print(pathway.is_multistep())  # isomeric transitions are not considered as steps
print(pathway.reduce())  # isomeric transitions can be compressed

Co59 -(n,g)-> Co60m -(IT)-> Co60
False
Co59 -(n,g)-> Co60


In [3]:
# Zaid data is contanied in the FispactZaid dataclass
zaid = pathway.parent
zaid

FispactZaid(element='Co', isotope=59, metastable=False)

In [4]:
# Finally, it is also possible to get a dataframe summarizing all the pathways
pathways_collection.to_dataframe().iloc[20:30]

Parent Intermediates       Reactions
Daughter % contribution                                     
Fe55     7.123            Fe54            []         [(n,g)]
         86.318           Fe56            []        [(n,2n)]
         6.119            Ni58            []         [(n,a)]
Fe59     10.285           Ni62            []         [(n,a)]
         3.422            Co59            []         [(n,p)]
         86.129           Fe58            []         [(n,g)]
Co57     96.742           Ni58            []        [(n,np)]
         3.067            Ni58        [Ni57]  [(n,2n), (b+)]
Co58     44.714           Ni58       [Co58m]   [(n,p), (IT)]
         54.559           Ni58            []         [(n,p)]

## Parsing of the output file at large

When parsing a FISPACT output the code requires to specify labels to be associated with the different cooling times after shutdown. No internal check is performed to assert that the number of labels provided corresponds to the number of cooling times defined in the FISPACT run. Simply the last N times will be selected where N is the number of provided labels.

In [1]:
from f4enix.output.fispact_legacy_out import FispactOutput

cooling_time_labels = ['24h', '1y']
outp = FispactOutput('testSS.out', cooling_time_labels)
# the original pypact TimeStep objects can be found in the inventory_data attribute
print(type(outp.inventory_data[0]))

<class 'pypact.output.timestep.TimeStep'>


The main attributes of the object are the SDDR and decay heat global dataframe and the pathways collection

In [6]:
outp.sddr.head()

,element,isotope,state,dose,cooling time,isotope % dose,Cumulative dose sum
70,Co,60,,0.003954,24h,48.786745,48.786745
57,Mn,54,,0.003297,24h,40.680298,89.467043
68,Co,58,,0.000581,24h,7.168715,96.635758
126,Ta,182,,0.000147,24h,1.807602,98.443360
67,Co,57,,0.000103,24h,1.277043,99.720403


In [2]:
outp.decay_heat.head()

,element,isotope,state,heat,alpha_heat,beta_heat,gamma_heat,cooling time,isotope % heat,Cumulative heat sum
70,Co,60,,2.297000e-09,0.0,8.549000e-11,2.212000e-09,24h,44.509873,44.509873
57,Mn,54,,2.044000e-09,0.0,9.798000e-12,2.034000e-09,24h,39.607393,84.117266
68,Co,58,,3.837000e-10,0.0,1.303000e-11,3.707000e-10,24h,7.435106,91.552372
67,Co,57,,2.546000e-10,0.0,3.245000e-11,2.222000e-10,24h,4.933484,96.485856
126,Ta,182,,1.045000e-10,0.0,1.507000e-11,8.939000e-11,24h,2.024938,98.510794


In [7]:
outp.pathways_collection

A common task when running an activation study in preparation of a D1S calculation is to identify what are the main decay pathways that should be tracked during the simulation.The output objects allows to build a pandas dataframe of the contact dose (and pathways) at specific cooling times.

In [8]:
df = outp.filter_by_cum_dose(
    perc=95, # cap dose at 95% of the total dose
    label='24h', # the label of the cooling time to filter by
    add_pathways=True) # include the pathways in the output

df.set_index(['element', 'isotope', 'state'])

dose cooling time  isotope % dose  \
element isotope state                                          
Mn      54             0.003297          24h       40.680298   
Co      60             0.003954          24h       48.786745   
                       0.003954          24h       48.786745   
                       0.003954          24h       48.786745   
Mn      54             0.003297          24h       40.680298   
Co      60             0.003954          24h       48.786745   
        58             0.000581          24h        7.168715   
                       0.000581          24h        7.168715   

                       Cumulative dose sum                           pathway  \
element isotope state                                                          
Mn      54                       89.467043                Fe54 -(n,p)-> Mn54   
Co      60                       48.786745  Co59 -(n,g)-> Co60m -(IT)-> Co60   
                                 48.786745                Co59 -(n,g)-> Co60   
                                 48.786745  Ni60 -(n,p)-> Co60m -(IT)-> Co60   
Mn      54                       89.467043               Mn55 -(n,2n)-> Mn54   
Co      60                       48.786745                Ni60 -(n,p)-> Co60   
        58                       96.635758                Ni58 -(n,p)-> Co58   
                                 96.635758  Ni58 -(n,p)-> Co58m -(IT)-> Co58   

                       pathway % dose  
element isotope state                  
Mn      54                  30.680267  
Co      60                  16.553343  
                            13.229014  
                            10.645268  
Mn      54                   9.917857  
Co      60                   7.852227  
        58                   3.911179  
                             3.205419

Similarly, it is possible to filter also for decay heat contributors. Alpha, Beta and gamma heat columns refer to the daughter isotope, not to the specific pathway.

In [4]:
df = outp.filter_by_cum_heating(
    perc=95, # cap heat at 95% of the total heat
    label='1y', # the label of the cooling time to filter by
    add_pathways=True) # include the pathways in the output
df.set_index(['element', 'isotope', 'state'])

heat  alpha_heat     beta_heat    gamma_heat  \
element isotope state                                                         
Co      60             7.034000e-10         0.0  2.618000e-11  6.772000e-10   
                       7.034000e-10         0.0  2.618000e-11  6.772000e-10   
                       7.034000e-10         0.0  2.618000e-11  6.772000e-10   
                       7.034000e-10         0.0  2.618000e-11  6.772000e-10   

                      cooling time  isotope % heat  Cumulative heat sum  \
element isotope state                                                     
Co      60                      1y       98.575549            98.575549   
                                1y       98.575549            98.575549   
                                1y       98.575549            98.575549   
                                1y       98.575549            98.575549   

                                                pathway  pathway % heat  
element isotope state                                                    
Co      60             Co59 -(n,g)-> Co60m -(IT)-> Co60       33.446684  
                                     Co59 -(n,g)-> Co60       26.729746  
                       Ni60 -(n,p)-> Co60m -(IT)-> Co60       21.509185  
                                     Ni60 -(n,p)-> Co60       15.865735